In [1]:
import pandas as pd
import sqlite3

# 1. Load the CSV automatically handling spaces and columns
df = pd.read_csv('Sample - Superstore.csv', encoding='ISO-8859-1')

# Clean column names automatically so they don't break in SQL
df.columns = df.columns.str.replace(' ', '_').str.replace('-', '_')

# 2. Connect to a clean SQL database engine
conn = sqlite3.connect(':memory:')
df.to_sql('SampleSuperstore', conn, index=False, if_exists='replace')

# 3. Define a quick function to run our pure SQL queries cleanly
def run_sql(query):
    return display(pd.read_sql_query(query, conn))

print("CSV Loaded into SQL Engine Successfully!")

CSV Loaded into SQL Engine Successfully!


In [3]:
print("--- 1. Subquery: Orders Above Average Sales ---")
run_sql("""
SELECT Order_ID, Customer_ID, Product_ID, Sales
FROM SampleSuperstore
WHERE Sales > (SELECT AVG(Sales) FROM SampleSuperstore)
ORDER BY Sales DESC LIMIT 5;
""")

print("\n--- 2. CTE: Customer Total Aggregations ---")
run_sql("""
WITH CustomerAggregates AS (
    SELECT Customer_ID, Customer_Name, Segment,
           ROUND(SUM(Sales), 2) AS Total_Spend,
           COUNT(DISTINCT Order_ID) AS Order_Count
    FROM SampleSuperstore
    GROUP BY Customer_ID
)
SELECT * FROM CustomerAggregates WHERE Total_Spend > 5000 ORDER BY Total_Spend DESC LIMIT 5;
""")

print("--- 3. Window Function: Regional Rankings (Fixed) ---")
run_sql("""
WITH RegionalSales AS (
    SELECT Region, Product_ID, SUM(Sales) AS Total_Product_Sales
    FROM SampleSuperstore
    GROUP BY Region, Product_ID
),
RankedSales AS (
    SELECT Region, Product_ID, ROUND(Total_Product_Sales, 2) AS Revenue,
           RANK() OVER (PARTITION BY Region ORDER BY Total_Product_Sales DESC) AS Sales_Rank
    FROM RegionalSales
)
SELECT Region, Product_ID, Revenue, Sales_Rank
FROM RankedSales
WHERE Sales_Rank <= 3
LIMIT 12;
""")

--- 1. Subquery: Orders Above Average Sales ---


,Order_ID,Customer_ID,Product_ID,Sales
0,CA-2014-145317,SM-20320,TEC-MA-10002412,22638.480
1,CA-2016-118689,TC-20980,TEC-CO-10004722,17499.950
2,CA-2017-140151,RB-19360,TEC-CO-10004722,13999.960
3,CA-2017-127180,TA-21385,TEC-CO-10004722,11199.968
4,CA-2017-166709,HL-15040,TEC-CO-10004722,10499.970



--- 2. CTE: Customer Total Aggregations ---


,Customer_ID,Customer_Name,Segment,Total_Spend,Order_Count
0,SM-20320,Sean Miller,Home Office,25043.05,5
1,TC-20980,Tamara Chand,Corporate,19052.22,5
2,RB-19360,Raymond Buch,Consumer,15117.34,6
3,TA-21385,Tom Ashbrook,Home Office,14595.62,4
4,AB-10105,Adrian Barton,Consumer,14473.57,10


--- 3. Window Function: Regional Rankings (Fixed) ---


,Region,Product_ID,Revenue,Sales_Rank
0,Central,TEC-CO-10004722,17499.95,1
1,Central,TEC-MA-10000822,14279.92,2
2,Central,OFF-BI-10001120,11339.94,3
3,East,TEC-CO-10004722,30099.91,1
4,East,TEC-MA-10001047,14299.89,2
5,East,FUR-BO-10004834,11717.03,3
6,South,TEC-MA-10002412,22638.48,1
7,South,TEC-MA-10001127,11374.94,2
8,South,OFF-BI-10001359,8342.01,3
9,West,TEC-CO-10004722,13999.96,1
